In [1]:
# Pyomo is an algebraic modeling language for Python. The pyomo_pounce
# import side-effect registers the "pounce" interior-point NLP solver
# with pyo.SolverFactory; the wheel bundles the solver binary, so no
# system install is required.
import pyomo.environ as pyo
import pyomo_pounce  # noqa: F401

Sets

$\mathcal{I} = \{1, \ldots, N\}$ circles

Parameters

$r_i$ radius of circle $i \in \mathcal{I}$

Variables

$W, H \ge 0$ rectangle width and height \
$x_i, y_i \ge 0$ center of circle $i \in \mathcal{I}$

Objective and Constraints

\begin{gather}
 \min_{W, H, x, y} \; W \cdot H \;\;\textrm{(area)} \\
 \textrm{s.t.} \;\; r_i \le x_i \le W - r_i \;\; \forall i \in \mathcal{I} \\
 r_i \le y_i \le H - r_i \;\; \forall i \in \mathcal{I} \\
 (x_i - x_j)^2 + (y_i - y_j)^2 \ge (r_i + r_j)^2 \;\; \forall i < j \\
 W, H \ge 0
\end{gather}

The pairwise non-overlap constraint is *non-convex* (its feasible region is the complement of an open ball), and the bilinear objective $W \cdot H$ is also non-convex. The problem therefore typically admits multiple local optima &mdash; the solver's converged value depends on the initial guess for $(x_i, y_i)$.

In [2]:
# Small instance: five circles with varied radii. The initial-condition
# (x0, y0) values are a loose starting layout -- the solver compacts the
# rectangle from here.
data = {
    "circles": [1, 2, 3, 4, 5],
    "r":       {1: 1.0, 2: 2.0, 3: 1.0, 4: 3.0, 5: 1.0},
    "x0":      {1: 2.0, 2: 6.0, 3: 2.0, 4: 8.0, 5: 12.0},
    "y0":      {1: 2.0, 2: 2.0, 3: 5.0, 4: 7.0, 5: 2.0},
}

In [3]:
# Build the NLP model. ConcreteModel binds parameter values at
# construction time (as opposed to AbstractModel, which is parameterized
# and only instantiated later).
m = pyo.ConcreteModel()

circles = list(data["circles"])
m.I = pyo.Set(initialize=circles, ordered=True)

# Per-circle radius parameter.
m.r = pyo.Param(m.I, initialize={i: data["r"][i] for i in circles})

# Loose upper bound for W, H, and the circle centers. 2 * sum(r) is
# generous (any reasonable packing fits in that square) and keeps the
# interior-point solver well-conditioned by bounding the variable range.
sum_r = sum(data["r"][i] for i in circles)
big = max(2.0 * sum_r, 4.0 * max(data["r"][i] for i in circles))

# Decision variables. W, H are the rectangle dimensions; x_i, y_i are
# circle centers.
m.W = pyo.Var(bounds=(0.0, big), initialize=2.0 * sum_r)
m.H = pyo.Var(bounds=(0.0, big), initialize=2.0 * sum_r)
m.x = pyo.Var(m.I, bounds=(0.0, big))
m.y = pyo.Var(m.I, bounds=(0.0, big))

# Seed the centers with the user-supplied initial guess. Interior-point
# methods are sensitive to the starting point on non-convex problems --
# different initial guesses can land at different local minima.
for i in circles:
    m.x[i].value = data["x0"][i]
    m.y[i].value = data["y0"][i]

# Containment: each circle fits horizontally and vertically inside the
# rectangle. Equivalent to r_i <= x_i <= W - r_i (and similarly y).
m.x_upper = pyo.Constraint(m.I, rule=lambda m, i: m.x[i] + m.r[i] <= m.W)
m.x_lower = pyo.Constraint(m.I, rule=lambda m, i: m.x[i] >= m.r[i])
m.y_upper = pyo.Constraint(m.I, rule=lambda m, i: m.y[i] + m.r[i] <= m.H)
m.y_lower = pyo.Constraint(m.I, rule=lambda m, i: m.y[i] >= m.r[i])

# Pairwise non-overlap. Squared form so the constraint stays smooth at
# the boundary (sqrt would have a non-differentiable point at distance
# zero). This is the non-convex piece of the model.
def no_overlap(m, i, j):
    if i >= j:
        return pyo.Constraint.Skip
    return (m.x[i] - m.x[j])**2 + (m.y[i] - m.y[j])**2 \
           >= (m.r[i] + m.r[j])**2
m.no_overlap = pyo.Constraint(m.I, m.I, rule=no_overlap)

# Objective: minimize the rectangle's area. Bilinear W * H -- also
# non-convex, but the interior-point solver handles it fine.
m.area = pyo.Objective(expr=m.W * m.H, sense=pyo.minimize)

In [4]:
# Solve with pounce, an interior-point NLP solver from John Kitchin,
# distributed as a pip wheel via pyomo-pounce. The wheel bundles the
# solver binary, so SolverFactory finds it without any path lookup.
# tee=True streams the solver log to stdout so you can watch the
# interior-point iterations.
solver = pyo.SolverFactory("pounce")
results = solver.solve(m, tee=True)

# Optimal rectangle dimensions and per-circle centers.
print(f"\nOptimal rectangle: W = {pyo.value(m.W):.3f}, "
      f"H = {pyo.value(m.H):.3f}, area = {pyo.value(m.area):.3f}")
print(f"\n{'circle':>6}  {'r':>5}  {'x':>7}  {'y':>7}")
for i in circles:
    print(f"{i:>6}  "
          f"{data['r'][i]:>5.1f}  "
          f"{pyo.value(m.x[i]):>7.3f}  "
          f"{pyo.value(m.y[i]):>7.3f}")

Reading C:\Users\Devin\AppData\Local\Temp\tmpaqwefqtd.pyomo.nl...
Parsed 12 vars, 30 cons, jac_nnz=70, h_nnz=31 in 0.01s
******************************************************************************
This program contains POUNCE, a Rust port of Ipopt for nonlinear optimization.
 Released under the Eclipse Public License (EPL) â€” drop-in compatible with Ipopt.
         For more information visit https://github.com/jkitchin/pounce
******************************************************************************

This is POUNCE version 0.2.0, running with linear solver FERAL.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:       70
Number of nonzeros in Lagrangian Hessian.............:       31

Total number of variables............................:       12
                     variables with only lower bounds:        0
                variables with lower and upper bounds:       12
                     variables with 

   4    1.1882614e2 5.91e-2  5.61e0  -1.0  1.02e1  -1.0 7.67e-1 1.63e-1f  1
   5    1.1321275e2 9.19e-1  4.84e0  -1.0  1.13e1  -1.4 4.07e-1 1.43e-1f  1
   6    9.8880683e1  3.97e0  3.67e0  -1.0  5.09e1  -1.9 8.13e-1 3.36e-1f  1
   7    8.9928674e1  3.38e0  2.54e0  -1.0  1.75e1  -1.5  1.00e0 4.01e-1f  1
   8    8.6310656e1  4.48e0  2.44e0  -1.0  7.10e1  -2.0 3.36e-1 1.45e-1f  1
   9    8.5504626e1  4.27e0  2.70e0  -1.0  3.14e1  -2.4  1.00e0 4.78e-2f  1
iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
  10    7.8440086e1  5.27e0  2.96e0  -1.0  6.39e1    -  1.94e-1 5.05e-1f  1
  11    7.5002804e1  4.45e0  3.85e0  -1.0  3.09e1  -2.0 5.00e-1 2.11e-1f  1
  12    7.5230105e1  3.79e0  3.72e0  -1.0  3.98e1  -2.5 6.59e-2 1.49e-1f  1
  13    7.3742628e1  3.33e0  3.68e0  -1.0  2.95e1    -  6.71e-2 1.25e-1f  1
  14    7.2958240e1  3.06e0  4.95e0  -1.0  3.13e1  -2.1 5.16e-1 8.05e-2f  1
  15    7.2975899e1  2.93e0  1.15e0  -1.0  1.14e1  -1.6 8.66e-1 4.54e-2h  1
  16   


Optimal rectangle: W = 9.351, H = 7.464, area = 69.794

circle      r        x        y
     1    1.0    1.103    5.023
     2    2.0    2.000    2.000
     3    1.0    2.713    6.379
     4    3.0    6.351    4.464
     5    1.0    8.351    1.000
